In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad 

In [ ]:
tgfb = sc.read_h5ad('data/raw/Seurat_object_TGFB_Perturb_seq.h5ad')
tgfb.obs['target_pathway'] = 'TGFb'
print('tgfb', tgfb.X.max())
print(type(tgfb.X))

In [ ]:
tnfa = sc.read_h5ad('data/raw/Seurat_object_TNFA_Perturb_seq.h5ad')
tnfa.obs['target_pathway'] = 'TNFa'
print('tnfa', tnfa.X.max())
print(type(tnfa.X))

In [ ]:
adata = ad.concat([tgfb, tnfa])

In [ ]:
adata.write_h5ad('data/Jiang.h5ad')


In [2]:
jiang = sc.read_h5ad('data/Jiang.h5ad')

In [3]:
jiang

AnnData object with n_obs × n_vars = 623237 × 23083
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'cell_type', 'percent.mito', 'sample_ID', 'Batch_info', 'bc1_well', 'bc2_well', 'bc3_well', 'guide', 'gene', 'mixscale_score', 'target_pathway', 'n_counts'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'ensembl_id'
    uns: 'hvg'

In [4]:
import pandas as pd
import pickle

In [5]:
df = jiang.obs[['target_pathway', 'gene', 'mixscale_score']]

# Calculate the average mixscale_score for each gene within each target_pathway
avg_scores = df.groupby(['target_pathway', 'gene'])['mixscale_score'].mean().reset_index()

# Initialize an empty dictionary to store the results
top_genes_dict = {}

# For each target_pathway, get the top k genes (k = 1, 5, 10)
for target_pathway in avg_scores['target_pathway'].unique():
    # Filter rows for the current target_pathway
    pathway_data = avg_scores[avg_scores['target_pathway'] == target_pathway]
    
    # Sort by mixscale_score in descending order
    sorted_genes = pathway_data.sort_values(by='mixscale_score', ascending=False)
    
    # For each k = 1, 5, 10, get the top k genes
    for k in [1, 2, 3, 4, 5, 10]:
        top_k_genes = sorted_genes.head(k)['gene'].tolist()
        key = f"{target_pathway}_top_{k}"
        top_genes_dict[key] = top_k_genes + ['NT']

# Save the dictionary using pickle
with open('top_k_genes.pkl', 'wb') as f:
    pickle.dump(top_genes_dict, f)

